In [1]:
# ── Goodreads shelf scraper — Non-Western Fiction ──────────────────────────
import subprocess
import pandas as pd
import time
import re
import os
from bs4 import BeautifulSoup
from urllib.parse import urlencode
from pathlib import Path

REPO_ROOT   = Path().resolve().parents[1]
RAW_DIR     = REPO_ROOT / "Data" / "Raw" / "non_western_fantasy"
OUTPUT_PATH = RAW_DIR / "goodreads_raw.csv"
DESC_OUTPUT = RAW_DIR / "goodreads_with_descriptions.csv"

DELAY      = 3.0
DESC_DELAY = 2.0
SAVE_EVERY = 50

CURL_CMD = [
    "curl", "-s", "-L", "--max-time", "20",
    "-A", "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "-H", "Accept-Language: en-US,en;q=0.9",
    "-H", "Accept: text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
]

SHELVES = [
    "african-fantasy",
    "asian-fantasy",
    "indigenous-fantasy",
    "south-american-fantasy",
    "australian-fantasy",
    "middle-eastern-fantasy",
    "latin-american-fantasy",
    "afrofuturism",
    "african-science-fiction",
    "asian-science-fiction",
]

def curl_get(url):
    result = subprocess.run(CURL_CMD + [url], capture_output=True, text=True, timeout=25)
    return result.stdout

print(f"✅ Setup complete")
print(f"   Shelves to scrape: {len(SHELVES)}")
print(f"   Output: {OUTPUT_PATH}")

✅ Setup complete
   Shelves to scrape: 10
   Output: C:\Users\Ready2Use\Documents\my-folder\Ironhack-week10\Book-recommendations\Data\Raw\non_western_fantasy\goodreads_raw.csv


In [2]:
# ── Test connection ────────────────────────────────────────────────────────
html = curl_get("https://www.goodreads.com/shelf/show/african-fantasy?page=1")
print(f"HTML length: {len(html)}")

soup = BeautifulSoup(html, "html.parser")
print(f"Page title: {soup.title.get_text(strip=True) if soup.title else 'NONE - blocked'}")

# Test book extraction
books = soup.select("div.elementList")
print(f"Book elements found: {len(books)}")

if books:
    first = books[0]
    title  = first.select_one("a.bookTitle")
    author = first.select_one("span[itemprop='name']")
    rating = first.select_one("span.minirating")
    print(f"\nFirst book:")
    print(f"  Title:  {title.get_text(strip=True) if title else 'NOT FOUND'}")
    print(f"  Author: {author.get_text(strip=True) if author else 'NOT FOUND'}")
    print(f"  Rating: {rating.get_text(strip=True) if rating else 'NOT FOUND'}")
    print(f"  Href:   {title['href'] if title else 'NOT FOUND'}")

HTML length: 273780
Page title: African Fantasy Books
Book elements found: 51

First book:
  Title:  Children of Blood and Bone (Legacy of Orïsha, #1)
  Author: Tomi Adeyemi
  Rating: NOT FOUND
  Href:   /book/show/34728667-children-of-blood-and-bone


In [3]:
# ── Parser and scraper functions ───────────────────────────────────────────
def parse_shelf_page(html, shelf):
    soup = BeautifulSoup(html, "html.parser")
    records = []

    for div in soup.select("div.elementList"):
        title_tag  = div.select_one("a.bookTitle")
        author_tag = div.select_one("span[itemprop='name']")
        grey_tag   = div.select_one("span.greyText.smallText")
        cover_tag  = div.select_one("img")

        if not title_tag:
            continue

        title  = title_tag.get_text(strip=True)
        href   = title_tag["href"].split("?")[0]
        author = author_tag.get_text(strip=True) if author_tag else ""

        # Extract rating, num_ratings and year from grey text
        grey_text   = grey_tag.get_text(strip=True) if grey_tag else ""
        rating_m    = re.search(r'avg rating ([\d.]+)', grey_text)
        num_rating_m = re.search(r'([\d,]+)\s+ratings', grey_text)
        year_m      = re.search(r'published (\d{4})', grey_text)
        cover_url   = cover_tag["src"] if cover_tag and cover_tag.get("src") else ""

        records.append({
            "title":        title,
            "author":       author,
            "goodreads_url": f"https://www.goodreads.com{href}",
            "cover_url":    cover_url,
            "avg_rating":   float(rating_m.group(1)) if rating_m else None,
            "num_ratings":  int(num_rating_m.group(1).replace(",", "")) if num_rating_m else None,
            "year_published": int(year_m.group(1)) if year_m else None,
            "shelf":        shelf,
        })

    # Check for next page
    next_tag = soup.select_one("a.next_page")
    has_next = next_tag is not None

    return records, has_next


def scrape_shelf(shelf, max_pages=20):
    all_records = []
    page = 1

    while page <= max_pages:
        url  = f"https://www.goodreads.com/shelf/show/{shelf}?page={page}"
        html = curl_get(url)

        if not html or len(html) < 1000:
            print(f"  ⚠️  Empty response on page {page}")
            break

        records, has_next = parse_shelf_page(html, shelf)

        if not records:
            print(f"  No books on page {page} — stopping")
            break

        all_records.extend(records)
        print(f"  Page {page}: {len(records)} books (total: {len(all_records)})")

        if not has_next:
            break

        page += 1
        time.sleep(DELAY)

    return all_records

# ── Quick test ─────────────────────────────────────────────────────────────
print("Testing african-fantasy shelf...")
test_records = scrape_shelf("african-fantasy", max_pages=1)
print(f"\nFirst book:")
for k, v in test_records[0].items():
    print(f"  {k}: {v}")

Testing african-fantasy shelf...
  Page 1: 50 books (total: 50)

First book:
  title: Children of Blood and Bone (Legacy of Orïsha, #1)
  author: Tomi Adeyemi
  goodreads_url: https://www.goodreads.com/book/show/34728667-children-of-blood-and-bone
  cover_url: https://i.gr-assets.com/images/S/compressed.photo.goodreads.com/books/1516127989l/34728667._SX50_.jpg
  avg_rating: 4.1
  num_ratings: 253767
  year_published: 2018
  shelf: african-fantasy


In [5]:
# ── Launch browser ─────────────────────────────────────────────────────────
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

options = webdriver.ChromeOptions()
options.add_argument("--window-size=1280,900")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)
driver.execute_script(
    "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
)
wait = WebDriverWait(driver, 20)

# ── Navigate to first shelf page ───────────────────────────────────────────
driver.get("https://www.goodreads.com/shelf/show/african-fantasy")
time.sleep(3)

print(f"Page title: {driver.title}")
print(f"URL: {driver.current_url}")

Page title: Sign in
URL: https://www.goodreads.com/user/sign_in?returnurl=%2Fshelf%2Fshow%2Fafrican-fantasy


In [ ]:
# ── Login to Goodreads ─────────────────────────────────────────────────────
from dotenv import load_dotenv
load_dotenv()

GR_EMAIL    = os.getenv("GOODREADS_EMAIL", "")
GR_PASSWORD = os.getenv("GOODREADS_PASSWORD", "")

driver.get("https://www.goodreads.com/user/sign_in")
time.sleep(3)

try:
    # Click "Sign in with email"
    email_btn = wait.until(
        EC.element_to_be_clickable((By.XPATH, "//a[contains(text(),'Sign in with email')]"))
    )
    email_btn.click()
    time.sleep(2)

    email_field = wait.until(
        EC.presence_of_element_located((By.ID, "user_email"))
    )
    email_field.send_keys(GR_EMAIL)

    password_field = driver.find_element(By.ID, "user_password")
    password_field.send_keys(GR_PASSWORD)

    submit_btn = driver.find_element(By.NAME, "next")
    submit_btn.click()
    time.sleep(3)

    if "sign_in" not in driver.current_url:
        print("✅ Login successful")
    else:
        print("⚠️  Login may have failed — check browser window")

except Exception as e:
    print(f"❌ Error: {e}")
    print("Please log in manually in the browser window")

In [ ]:
# ── Navigate to login page and log in manually ─────────────────────────────
driver.get("https://www.goodreads.com/user/sign_in")
time.sleep(3)

print("⏳ Please log in to Goodreads manually in the browser window.")
print("   Once logged in come back and run the verify cell.")

In [6]:
# ── Verify login ───────────────────────────────────────────────────────────
if "sign_in" not in driver.current_url:
    print("✅ Logged in — ready to scrape")
else:
    page_text = driver.page_source.lower()
    if "my books" in page_text or "profile" in page_text:
        print("✅ Logged in — ready to scrape")
    else:
        print("⚠️  Not logged in yet — please log in and rerun this cell")

✅ Logged in — ready to scrape


In [7]:
# ── Test pagination after login ────────────────────────────────────────────
driver.get("https://www.goodreads.com/shelf/show/african-fantasy")
time.sleep(3)

soup = BeautifulSoup(driver.page_source, "html.parser")
valid_books = [b for b in soup.select("div.elementList") if b.select_one("a.bookTitle")]
print(f"Books on page 1: {len(valid_books)}")

next_btn = soup.select_one("a.next_page")
print(f"Next page link: {next_btn}")

Books on page 1: 50
Next page link: <a class="next_page" href="/shelf/show/african-fantasy?page=2" rel="next">next »</a>


In [ ]:
# ── Test page 2 ────────────────────────────────────────────────────────────
# Click next button
next_btn = driver.find_element(By.CSS_SELECTOR, "a.next_page")
driver.execute_script("arguments[0].click();", next_btn)
time.sleep(3)

soup2 = BeautifulSoup(driver.page_source, "html.parser")
valid_books2 = [b for b in soup2.select("div.elementList") if b.select_one("a.bookTitle")]
print(f"Books on page 2: {len(valid_books2)}")
print(f"First book: {valid_books2[0].select_one('a.bookTitle').get_text(strip=True)}")
print(f"Last book:  {valid_books2[-1].select_one('a.bookTitle').get_text(strip=True)}")

next_btn2 = soup2.select_one("a.next_page")
print(f"Next page link: {next_btn2}")

In [8]:
# ── Selenium-based shelf scraper ───────────────────────────────────────────
def parse_shelf_page_selenium(shelf):
    soup = BeautifulSoup(driver.page_source, "html.parser")
    records = []

    for div in soup.select("div.elementList"):
        title_tag  = div.select_one("a.bookTitle")
        author_tag = div.select_one("span[itemprop='name']")
        grey_tag   = div.select_one("span.greyText.smallText")
        cover_tag  = div.select_one("img")

        if not title_tag:
            continue

        title  = title_tag.get_text(strip=True)
        href   = title_tag["href"].split("?")[0]
        author = author_tag.get_text(strip=True) if author_tag else ""

        grey_text    = grey_tag.get_text(strip=True) if grey_tag else ""
        rating_m     = re.search(r'avg rating ([\d.]+)', grey_text)
        num_rating_m = re.search(r'([\d,]+)\s+ratings', grey_text)
        year_m       = re.search(r'published (\d{4})', grey_text)
        cover_url    = cover_tag["src"] if cover_tag and cover_tag.get("src") else ""

        records.append({
            "title":          title,
            "author":         author,
            "goodreads_url":  f"https://www.goodreads.com{href}",
            "cover_url":      cover_url,
            "avg_rating":     float(rating_m.group(1)) if rating_m else None,
            "num_ratings":    int(num_rating_m.group(1).replace(",", "")) if num_rating_m else None,
            "year_published": int(year_m.group(1)) if year_m else None,
            "shelf":          shelf,
        })

    return records


def scrape_shelf_selenium(shelf, max_pages=20):
    all_records = []
    url = f"https://www.goodreads.com/shelf/show/{shelf}"
    driver.get(url)
    time.sleep(3)
    page = 1

    while page <= max_pages:
        records = parse_shelf_page_selenium(shelf)

        if not records:
            print(f"  No books on page {page} — stopping")
            break

        all_records.extend(records)
        print(f"  Page {page}: {len(records)} books (total: {len(all_records)})")

        # Check for next button
        try:
            next_btn = driver.find_element(By.CSS_SELECTOR, "a.next_page")
            driver.execute_script("arguments[0].click();", next_btn)
            time.sleep(3)
            page += 1
        except:
            print(f"  No more pages")
            break

    return all_records

# ── Quick test ─────────────────────────────────────────────────────────────
print("Testing african-fantasy...")
test = scrape_shelf_selenium("african-fantasy", max_pages=2)
print(f"\nTotal: {len(test)} books")
print(f"Page 1 first: {test[0]['title']}")
print(f"Page 2 first: {test[50]['title'] if len(test) > 50 else 'N/A'}")

Testing african-fantasy...
  Page 1: 50 books (total: 50)
  Page 2: 50 books (total: 100)

Total: 100 books
Page 1 first: Children of Blood and Bone (Legacy of Orïsha, #1)
Page 2 first: The Merciless Ones (Deathless, #2)


In [9]:
# ── Full scraping loop ─────────────────────────────────────────────────────
all_records = []

for shelf in SHELVES:
    print(f"\n── Shelf: {shelf} ──")
    try:
        records = scrape_shelf_selenium(shelf, max_pages=20)
        print(f"   ✅ {len(records)} books collected")
        all_records.extend(records)
    except Exception as e:
        print(f"   ❌ Failed: {e}")
    time.sleep(DELAY)

# Deduplicate by title + author
df = pd.DataFrame(all_records)
df = df.drop_duplicates(subset=["title", "author"])
df = df.reset_index(drop=True)

print(f"\n✅ Scraping complete")
print(f"   Raw results:  {len(all_records):,}")
print(f"   After dedup:  {len(df):,}")
print(f"\n   Shelf breakdown:")
for shelf, count in df["shelf"].value_counts().items():
    print(f"     {shelf}: {count}")

df.to_csv(OUTPUT_PATH, index=False)
print(f"\n✅ Saved to {OUTPUT_PATH}")


── Shelf: african-fantasy ──
  Page 1: 50 books (total: 50)
  Page 2: 50 books (total: 100)
  Page 3: 50 books (total: 150)
  Page 4: 50 books (total: 200)
  Page 5: 21 books (total: 221)
  No more pages
   ✅ 221 books collected

── Shelf: asian-fantasy ──
  Page 1: 50 books (total: 50)
  Page 2: 50 books (total: 100)
  Page 3: 50 books (total: 150)
  Page 4: 50 books (total: 200)
  Page 5: 50 books (total: 250)
  Page 6: 50 books (total: 300)
  Page 7: 50 books (total: 350)
  Page 8: 50 books (total: 400)
  Page 9: 50 books (total: 450)
  Page 10: 50 books (total: 500)
  No more pages
   ✅ 500 books collected

── Shelf: indigenous-fantasy ──
  Page 1: 31 books (total: 31)
  No more pages
   ✅ 31 books collected

── Shelf: south-american-fantasy ──
  Page 1: 8 books (total: 8)
  No more pages
   ✅ 8 books collected

── Shelf: australian-fantasy ──
  Page 1: 42 books (total: 42)
  No more pages
   ✅ 42 books collected

── Shelf: middle-eastern-fantasy ──
  Page 1: 50 books (total: 50)


In [10]:
# ── Description enrichment ─────────────────────────────────────────────────
def fetch_description(goodreads_url):
    driver.get(goodreads_url)
    time.sleep(2)
    soup = BeautifulSoup(driver.page_source, "html.parser")

    # Try modern selector first, fall back to legacy
    desc = soup.select_one("div[data-testid='description'] span.Formatted")
    if not desc:
        desc = soup.select_one("#description span")
    if not desc:
        desc = soup.select_one("span.Formatted")

    return desc.get_text(separator=" ", strip=True) if desc else ""

# ── Quick test ─────────────────────────────────────────────────────────────
test_url = df["goodreads_url"].iloc[0]
print(f"Testing: {test_url}")
desc = fetch_description(test_url)
print(f"Description: {desc[:300]}")

Testing: https://www.goodreads.com/book/show/34728667-children-of-blood-and-bone
Description: They killed my mother. They took our magic. They tried to bury us. Now we rise. Zélie Adebola remembers when the soil of Orïsha hummed with magic. Burners ignited flames, Tiders beckoned waves, and Zélie’s Reaper mother summoned forth souls. But everything changed the night magic disappeared. Under 


In [11]:
# ── Full description enrichment loop ──────────────────────────────────────
DESC_CKPT_PATH = RAW_DIR / "goodreads_desc_checkpoint.csv"

# Resume from checkpoint if exists
if DESC_CKPT_PATH.exists():
    df_desc = pd.read_csv(DESC_CKPT_PATH)
    done_urls = set(df_desc.loc[df_desc["description"].notna() & 
                   (df_desc["description"] != ""), "goodreads_url"])
    print(f"Resuming — {len(done_urls):,} already done")
else:
    df_desc = df.copy()
    df_desc["description"] = ""
    done_urls = set()
    print("Starting fresh")

todo = df_desc[~df_desc["goodreads_url"].isin(done_urls)]
print(f"Books to fetch: {len(todo):,}")

for i, (idx, row) in enumerate(todo.iterrows(), 1):
    try:
        desc = fetch_description(row["goodreads_url"])
        df_desc.at[idx, "description"] = desc
        status = "✅" if desc else "⚠️"
    except Exception as e:
        df_desc.at[idx, "description"] = ""
        status = "❌"

    if i % 10 == 0:
        print(f"  [{i}/{len(todo)}] {status} {row['title'][:40]}")

    if i % SAVE_EVERY == 0:
        df_desc.to_csv(DESC_CKPT_PATH, index=False)
        print(f"  >> Checkpoint saved ({i} done)")

    time.sleep(DESC_DELAY)

df_desc.to_csv(DESC_OUTPUT, index=False)
print(f"\n✅ Done — {len(df_desc):,} books")
print(f"   Has description: {(df_desc['description'] != '').sum():,}")

Resuming — 1,433 already done
Books to fetch: 696
  [10/696] ⚠️ The Song of Wirrun: Ice is Coming, Dark 
  [20/696] ✅ The Intuitionist (Paperback)
  [30/696] ✅ Black Panther: Long Live the King (Paper
  [40/696] ✅ Destroyer of Light (Hardcover)
  [50/696] ✅ The Comet (ebook)
  >> Checkpoint saved (50 done)
  [60/696] ✅ Nova (Paperback)
  [70/696] ✅ Last Gate of the Emperor (Hardcover)
  [80/696] ✅ Black Panther, Vol. 7: The Intergalactic
  [90/696] ✅ Critique of Black Reason (Paperback)
  [100/696] ✅ The Ephemera Collector (Hardcover)
  >> Checkpoint saved (100 done)
  [110/696] ✅ No Gods, No Monsters (Convergence Saga, 
  [120/696] ✅ Amari and the Night Brothers (Supernatur
  [130/696] ✅ The Jewels of Aptor (Paperback)
  [140/696] ✅ SUN RA (Paperback)
  [150/696] ✅ Unraveling (Paperback)
  >> Checkpoint saved (150 done)
  [160/696] ✅ Black Panther: The Young Prince (Hardcov
  [170/696] ✅ The Awakened Kingdom  (The Inheritance T
  [180/696] ✅ Bitch Planet, Vol. 2: President Bitch (P
  

In [12]:
import json, pandas as pd
from pathlib import Path

REPO_ROOT = Path().resolve().parents[1]
RAW_DIR   = REPO_ROOT / "Data" / "Raw" / "non_western_fantasy"

# ── Load both ──────────────────────────────────────────────────────────────
ol = pd.DataFrame(json.load(open(RAW_DIR / "ol_genre_first.json")))
gr = pd.read_csv(RAW_DIR / "goodreads_with_descriptions.csv")

print(f"OL shape:         {ol.shape}")
print(f"Goodreads shape:  {gr.shape}")

# ── OL field coverage ──────────────────────────────────────────────────────
print("\n── OL field coverage ──")
for col in ["title","author","year_published","cover_url","avg_rating","num_ratings","subjects"]:
    filled = ol[col].apply(lambda x: bool(x) if not isinstance(x, float) else pd.notna(x)).sum()
    print(f"  {col:20s}: {filled:,}/{len(ol):,} ({filled/len(ol)*100:.0f}%)")

# ── Goodreads field coverage ───────────────────────────────────────────────
print("\n── Goodreads field coverage ──")
for col in ["title","author","description","cover_url","avg_rating","num_ratings","year_published"]:
    if col in gr.columns:
        filled = gr[col].notna().sum()
        print(f"  {col:20s}: {filled:,}/{len(gr):,} ({filled/len(gr)*100:.0f}%)")

# ── Goodreads shelf breakdown ──────────────────────────────────────────────
print("\n── Goodreads shelf breakdown ──")
print(gr["shelf"].value_counts().to_string())

# ── OL source_tag breakdown ────────────────────────────────────────────────
print("\n── OL source_tag breakdown ──")
print(ol["source_tag"].value_counts().to_string())

OL shape:         (4364, 12)
Goodreads shape:  (2129, 9)

── OL field coverage ──
  title               : 4,364/4,364 (100%)
  author              : 4,255/4,364 (98%)
  year_published      : 4,305/4,364 (99%)
  cover_url           : 2,446/4,364 (56%)
  avg_rating          : 759/4,364 (17%)
  num_ratings         : 759/4,364 (17%)
  subjects            : 3,899/4,364 (89%)

── Goodreads field coverage ──
  title               : 2,129/2,129 (100%)
  author              : 2,129/2,129 (100%)
  description         : 2,108/2,129 (99%)
  cover_url           : 2,129/2,129 (100%)
  avg_rating          : 1,996/2,129 (94%)
  num_ratings         : 1,983/2,129 (93%)
  year_published      : 1,713/2,129 (80%)

── Goodreads shelf breakdown ──
shelf
asian-fantasy              988
afrofuturism               741
african-fantasy            221
middle-eastern-fantasy      57
australian-fantasy          42
indigenous-fantasy          31
asian-science-fiction       23
latin-american-fantasy      17
south-ameri